In [ ]:
!pip install captum -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 12.2 MB/s eta 0:00:00


In [ ]:

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind
from captum.attr import LayerGradCam

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [ ]:
# CIFAR-10 channel statistics (computed over training set).
transform_norm = transforms.Normalize(
    mean=(0.4914, 0.4822, 0.4465),
    std=(0.2023, 0.1994, 0.2010)
)

clean_transform = transforms.Compose([
    transforms.ToTensor(),
    transform_norm
])

# DETERMINISTIC shift: degrees=(45,45) forces exactly +45° every time.
# RandomRotation(45) samples uniformly from [-45, 45], so test_dataset_shift[i]
# returns a *different* image on every call — corrupting D(k) measurements.
shift_transform = transforms.Compose([
    transforms.RandomRotation(degrees=(45, 45)),
    transforms.GaussianBlur(5),
    transforms.ToTensor(),
    transform_norm
])

In [ ]:
BATCH_SIZE = 128

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=clean_transform
)
test_dataset_clean = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=clean_transform
)
test_dataset_shift = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=shift_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
test_loader_clean = torch.utils.data.DataLoader(
    test_dataset_clean, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)
test_loader_shift = torch.utils.data.DataLoader(
    test_dataset_shift, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

classes = ("plane", "car", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

100%|██████████| 170M/170M [24:52<00:00, 114kB/s]


In [ ]:
# CIFAR-10 adapted ResNet18: 3×3 stem conv, no maxpool (images are 32×32).
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, 10)
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}  Loss={running_loss/len(train_loader):.4f}")


Epoch 1/10  Loss=1.1985
Epoch 2/10  Loss=0.7135
Epoch 3/10  Loss=0.5235
Epoch 4/10  Loss=0.3936
Epoch 5/10  Loss=0.2895
Epoch 6/10  Loss=0.2046
Epoch 7/10  Loss=0.1485
Epoch 8/10  Loss=0.1064
Epoch 9/10  Loss=0.0819
Epoch 10/10  Loss=0.0789


In [ ]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(model(images), 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

clean_accuracy   = evaluate_model(model, test_loader_clean, device)
shifted_accuracy = evaluate_model(model, test_loader_shift, device)
print(f"Accuracy on clean test set:   {clean_accuracy:.2f}%")
print(f"Accuracy on shifted test set: {shifted_accuracy:.2f}%")


Accuracy on clean test set:   80.31%
Accuracy on shifted test set: 16.70%


In [ ]:
misclassified_shift = []

model.eval()
with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader_shift):
        images, labels = images.to(device), labels.to(device)
        _, predicted = torch.max(model(images), 1)
        for j in range(len(labels)):
            if predicted[j] != labels[j]:
                dataset_idx = batch_idx * BATCH_SIZE + j
                if dataset_idx < len(test_dataset_shift):
                    misclassified_shift.append(
                        (dataset_idx, labels[j].item(), predicted[j].item())
                    )

print(f"Found {len(misclassified_shift)} misclassified samples in the shifted test set.")


Found 8328 misclassified samples in the shifted test set.


In [ ]:
gradcam_layers = [
    LayerGradCam(model, model.layer1[0].conv1),
    LayerGradCam(model, model.layer1[1].conv2),
    LayerGradCam(model, model.layer2[0].conv1),
    LayerGradCam(model, model.layer2[1].conv2),
    LayerGradCam(model, model.layer3[0].conv1),
    LayerGradCam(model, model.layer3[1].conv2),
    LayerGradCam(model, model.layer4[0].conv1),
    LayerGradCam(model, model.layer4[1].conv2),
]
N_LAYERS = len(gradcam_layers)

layer_labels = [
    "L1-B1-C1", "L1-B2-C2",
    "L2-B1-C1", "L2-B2-C2",
    "L3-B1-C1", "L3-B2-C2",
    "L4-B1-C1", "L4-B2-C2",
]


In [ ]:
def compute_all_dk(gradcam_layers, clean_image, shifted_image, target_class):
    """
    Returns D(k) = ||GradCAM_shifted(k) - GradCAM_clean(k)|| for each layer k.

    Accepts unbatched (C, H, W) tensors; batch dim is added internally.
    Call twice per sample — once with target_class=true_label (drift of the
    correct semantic path) and once with target_class=pred_label (drift of
    the model's own mistaken reasoning path).
    """
    clean_t   = clean_image.unsqueeze(0).to(device).requires_grad_(True)
    shifted_t = shifted_image.unsqueeze(0).to(device).requires_grad_(True)

    dk_values = []
    for layer_gc in gradcam_layers:
        attr_clean   = layer_gc.attribute(clean_t,   target=target_class)
        attr_shifted = layer_gc.attribute(shifted_t, target=target_class)
        dk_values.append((attr_shifted - attr_clean).norm().item())
    return dk_values

In [ ]:
N_SAMPLES = 500

all_scores_true = [[] for _ in range(N_LAYERS)]
all_scores_pred = [[] for _ in range(N_LAYERS)]

for dataset_idx, true_label, pred_label in misclassified_shift[:N_SAMPLES]:
    clean_img,   _ = test_dataset_clean[dataset_idx]   # (C, H, W)
    shifted_img, _ = test_dataset_shift[dataset_idx]   # (C, H, W) — deterministic

    dk_true = compute_all_dk(gradcam_layers, clean_img, shifted_img, true_label)
    dk_pred = compute_all_dk(gradcam_layers, clean_img, shifted_img, pred_label)

    for k in range(N_LAYERS):
        all_scores_true[k].append(dk_true[k])
        all_scores_pred[k].append(dk_pred[k])

# ── Save raw scores — future-you will be grateful ─────────────────────────────
np.save("dk_true.npy", np.array(all_scores_true))   # shape: (N_LAYERS, N_SAMPLES)
np.save("dk_pred.npy", np.array(all_scores_pred))
print("Raw D(k) arrays saved to dk_true.npy and dk_pred.npy")


Raw D(k) arrays saved to dk_true.npy and dk_pred.npy


In [ ]:
def summarise(scores):
    """Returns (mean, std, ci_low, ci_high) arrays across layers."""
    means, stds, ci_lo, ci_hi = [], [], [], []
    for layer_scores in scores:
        a = np.array(layer_scores)
        m = np.mean(a)
        s = np.std(a, ddof=1)
        n = len(a)
        se = s / np.sqrt(n)
        t_crit = stats.t.ppf(0.975, df=n - 1)
        means.append(m)
        stds.append(s)
        ci_lo.append(m - t_crit * se)
        ci_hi.append(m + t_crit * se)
    return (np.array(means), np.array(stds),
            np.array(ci_lo), np.array(ci_hi))

avg_true, std_true, ci_lo_true, ci_hi_true = summarise(all_scores_true)
avg_pred, std_pred, ci_lo_pred, ci_hi_pred = summarise(all_scores_pred)

print("\nD(k) — drift relative to TRUE label  [mean ± std  (95% CI)]:")
for k in range(N_LAYERS):
    print(f"  D({k+1:}) = {avg_true[k]:.4f} ± {std_true[k]:.4f}"
          f"  CI=[{ci_lo_true[k]:.4f}, {ci_hi_true[k]:.4f}]")

print("\nD(k) — drift relative to PREDICTED label  [mean ± std  (95% CI)]:")
for k in range(N_LAYERS):
    print(f"  D({k+1}) = {avg_pred[k]:.4f} ± {std_pred[k]:.4f}"
          f"  CI=[{ci_lo_pred[k]:.4f}, {ci_hi_pred[k]:.4f}]")



D(k) — drift relative to TRUE label  [mean ± std  (95% CI)]:
  D(1) = 0.1327 ± 0.0484  CI=[0.1284, 0.1369]
  D(2) = 0.1290 ± 0.0437  CI=[0.1252, 0.1329]
  D(3) = 0.2892 ± 0.1137  CI=[0.2792, 0.2992]
  D(4) = 0.2990 ± 0.0832  CI=[0.2917, 0.3063]
  D(5) = 0.6142 ± 0.1660  CI=[0.5997, 0.6288]
  D(6) = 0.5922 ± 0.1871  CI=[0.5758, 0.6087]
  D(7) = 1.1761 ± 0.4179  CI=[1.1394, 1.2128]
  D(8) = 2.0936 ± 1.1916  CI=[1.9889, 2.1983]

D(k) — drift relative to PREDICTED label  [mean ± std  (95% CI)]:
  D(1) = 0.1075 ± 0.0475  CI=[0.1033, 0.1116]
  D(2) = 0.1115 ± 0.0468  CI=[0.1074, 0.1156]
  D(3) = 0.2570 ± 0.1159  CI=[0.2468, 0.2672]
  D(4) = 0.2467 ± 0.0747  CI=[0.2401, 0.2533]
  D(5) = 0.5668 ± 0.1507  CI=[0.5535, 0.5800]
  D(6) = 0.5150 ± 0.1822  CI=[0.4990, 0.5310]
  D(7) = 0.9655 ± 0.2738  CI=[0.9415, 0.9896]
  D(8) = 1.4267 ± 0.8242  CI=[1.3543, 1.4991]


In [ ]:
growth_true = [avg_true[i] / avg_true[i-1] for i in range(1, N_LAYERS)]
growth_pred = [avg_pred[i] / avg_pred[i-1] for i in range(1, N_LAYERS)]

print("\nLocality Growth Ratios L(k) — true label:")
for i, r in enumerate(growth_true):
    print(f"  L({i+2}) = {r:.3f}")

print("\nLocality Growth Ratios L(k) — predicted label:")
for i, r in enumerate(growth_pred):
    print(f"  L({i+2}) = {r:.3f}")

# ── Layer-to-layer t-tests: does D(k) significantly exceed D(k-1)? ────────────
# Tests the core claim ("deeper layers have larger drift") rather than
# asserting it from a plot. A p-value lets you write "D(5) significantly
# exceeds D(4) (p=0.003)" instead of "the graph looks bigger".
print("\nLayer-to-layer t-tests — TRUE label drift:")
for k in range(1, N_LAYERS):
    t, p = ttest_ind(all_scores_true[k], all_scores_true[k-1])
    sig = "✓" if p < 0.05 else "✗"
    print(f"  D({k+1}) vs D({k}): t={t:.3f}  p={p:.4f}  {sig}")

print("\nLayer-to-layer t-tests — PREDICTED label drift:")
for k in range(1, N_LAYERS):
    t, p = ttest_ind(all_scores_pred[k], all_scores_pred[k-1])
    sig = "✓" if p < 0.05 else "✗"
    print(f"  D({k+1}) vs D({k}): t={t:.3f}  p={p:.4f}  {sig}")



Locality Growth Ratios L(k) — true label:
  L(2) = 0.972
  L(3) = 2.241
  L(4) = 1.034
  L(5) = 2.054
  L(6) = 0.964
  L(7) = 1.986
  L(8) = 1.780

Locality Growth Ratios L(k) — predicted label:
  L(2) = 1.038
  L(3) = 2.305
  L(4) = 0.960
  L(5) = 2.297
  L(6) = 0.909
  L(7) = 1.875
  L(8) = 1.478

Layer-to-layer t-tests — TRUE label drift:
  D(2) vs D(1): t=-1.256  p=0.2094  ✗
  D(3) vs D(2): t=29.403  p=0.0000  ✓
  D(4) vs D(3): t=1.563  p=0.1185  ✗
  D(5) vs D(4): t=37.965  p=0.0000  ✓
  D(6) vs D(5): t=-1.969  p=0.0493  ✓
  D(7) vs D(6): t=28.514  p=0.0000  ✓
  D(8) vs D(7): t=16.247  p=0.0000  ✓

Layer-to-layer t-tests — PREDICTED label drift:
  D(2) vs D(1): t=1.352  p=0.1767  ✗
  D(3) vs D(2): t=26.028  p=0.0000  ✓
  D(4) vs D(3): t=-1.673  p=0.0946  ✗
  D(5) vs D(4): t=42.547  p=0.0000  ✓
  D(6) vs D(5): t=-4.895  p=0.0000  ✓
  D(7) vs D(6): t=30.632  p=0.0000  ✓
  D(8) vs D(7): t=11.873  p=0.0000  ✓


In [ ]:
ax = axes[0]
err_true = [avg_true - ci_lo_true, ci_hi_true - avg_true]
err_pred = [avg_pred - ci_lo_pred, ci_hi_pred - avg_pred]
ax.errorbar(depth, avg_true, yerr=err_true,
            marker="o", linewidth=2, capsize=5, label="Target = true label")
ax.errorbar(depth, avg_pred, yerr=err_pred,
            marker="s", linewidth=2, capsize=5, linestyle="--",
            label="Target = predicted label")
ax.set_xticks(depth)
ax.set_xticklabels(layer_labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("D(k)")
ax.set_title("Attribution Drift D(k)\n(mean ± 95% CI, n=500)")
ax.legend()
ax.grid(True, alpha=0.4)

In [ ]:
ax = axes[1]
ax.plot(depth_ratio, growth_true, marker="o", linewidth=2,
        label="Target = true label")
ax.plot(depth_ratio, growth_pred, marker="s", linewidth=2, linestyle="--",
        label="Target = predicted label")
ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.2,
           label="L(k)=1  (no amplification)")
ax.set_xticks(depth_ratio)
ax.set_xticklabels(layer_labels[1:], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("L(k) = D(k) / D(k−1)")
ax.set_title("Locality Growth Ratio L(k)\n(Domino Effect core metric)")
ax.legend()
ax.grid(True, alpha=0.4)


In [ ]:
# This is where the interesting result lives: if the predicted-label curve
# amplifies more aggressively than the true-label curve, failure trajectories
# are more drift-prone than correct semantic trajectories.
ax = axes[2]
x = np.arange(N_LAYERS)
w = 0.35
ax.bar(x - w/2, avg_true, w, yerr=std_true, capsize=3,
       label="True label", alpha=0.8)
ax.bar(x + w/2, avg_pred, w, yerr=std_pred, capsize=3,
       label="Predicted label", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(layer_labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("D(k)")
ax.set_title("True vs Predicted Drift per Layer\n(bar = mean, whisker = std)")
ax.legend()
ax.grid(True, alpha=0.4, axis="y")

plt.tight_layout()
plt.savefig("domino_effect_resnet18.png", dpi=150)
plt.show()
print("Plot saved to domino_effect_resnet18.png")

<Figure size 640x480 with 0 Axes>

Plot saved to domino_effect_resnet18.png
